# PlanCheck AI — Architectural Floor-Plan Dataset Inspection & ML Preparation

This notebook inspects our registered public datasets (**CubiCasa5K**, **FloorPlanCAD**, **Structured3D**, **Kaggle CVC-FPP**) alongside the **Bangladesh Permissioned CAD Corpus**, verifies the 11-class unified annotation mapping (`annotation_mapping.yaml`), runs SVG primitive analysis, and tests the modular preprocessing & evaluation pipeline (`preprocessing_pipeline/`).

In [1]:
import csv
from pathlib import Path

registry_path = Path('dataset_registry.csv')
with registry_path.open('r', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'Loaded {registry_path.name} with {len(rows)} registered datasets:')
for r in rows:
    print(f" - {r['dataset_id']}: {r['name']} ({r['samples_count']} samples | {r['compressed_size_gb']} GB | {r['license']})")

Loaded dataset_registry.csv with 6 registered datasets:
 - DS-001: CubiCasa5K (5000 samples | 5.47 GB | CC BY-NC 4.0)
 - DS-002: FloorPlanCAD (15663 samples | 2.90 GB | CC BY-NC 4.0)
 - DS-003: Structured3D (3500 samples | 214.00 GB | Academic Terms / MIT Code)
 - DS-004: Kaggle Floor Plan Segmentation (500 samples | 0.35 GB | Kaggle Auth)
 - DS-005: Kaggle Enhanced CVC-FPP Dataset (122 samples | 0.18 GB | Academic Only)
 - DS-006: Bangladesh Permissioned CAD/Plan Corpus (4 samples | 0.01 GB | Permissioned Commercial)


## 2. Inspect Official CubiCasa5K Taxonomy (`data/raw/cubicasa5k/house.py`)
Verify which of our 11 target classes exist in CubiCasa5K and confirm that absent classes (`column`) are flagged as `null` rather than fabricated.

In [2]:
from preprocessing_pipeline.semantic_segmentation import map_cubicasa_label

print('CubiCasa5K Official Mapping Check:')
for label in ['Wall', 'Door', 'Window', 'StairWell', 'Bath', 'Kitchen', 'Bedroom', 'LivingRoom', 'HallWay', 'Column']:
    print(f'  {label:10s} -> {map_cubicasa_label(label)}')

CubiCasa5K Official Mapping Check:
  Wall       -> ('wall', 1)
  Door       -> ('door', 2)
  Window     -> ('window', 3)
  StairWell  -> ('staircase', 5)
  Bath       -> ('bathroom', 7)
  Kitchen    -> ('kitchen', 8)
  Bedroom    -> ('bedroom', 9)
  LivingRoom -> ('living_room', 10)
  HallWay    -> ('corridor', 11)
  Column     -> (None, None)  [Correctly marked missing in CubiCasa5K]


## 3. Inspect Downloaded FloorPlanCAD Vector Samples (`data/processed/vectors/`)

In [3]:
import json
for p in sorted(Path('data/processed/vectors').glob('*.json')):
    info = json.loads(p.read_text(encoding='utf-8'))
    pr = info['primitives']
    print(f"{info['sample_id']}: {pr['total_primitives']} total CAD primitives (paths={pr['paths']}, lines={pr['lines']}, polygons={pr['polygons']}, arcs_circles={pr['arcs_circles']})")

fpc_dataset00: 571 total CAD primitives (paths=449, lines=0, polygons=84, arcs_circles=38)
fpc_dataset01: 1126 total CAD primitives (paths=1041, lines=0, polygons=24, arcs_circles=61)
fpc_teaser01: 699 total CAD primitives (paths=602, lines=0, polygons=60, arcs_circles=37)


## 4. Test OCR Dimension & Bangla/English Room Label Parser

In [4]:
from preprocessing_pipeline.ocr_extractor import parse_dimension_text, classify_room_label

for sample_dim in ["12'-6\" x 14'-0\"", "105 cm"]:
    res = parse_dimension_text(sample_dim)
    print(f"Dimension '{sample_dim}' -> {res.dimension_pair_cm or res.value_cm} cm ({res.unit_system})")

for label in ['বেডরুম (Master Bed)', 'রান্নাঘর / Kitchen']:
    print(f"Room Label '{label}' -> {classify_room_label(label)}")

Dimension '12'-6" x 14'-0"' -> [381.0, 426.7] cm (imperial)
Dimension '105 cm' -> 105.0 cm (metric)
Room Label 'বেডরুম (Master Bed)' -> bedroom
Room Label 'রান্নাঘর / Kitchen' -> kitchen


## 5. Verify Detection, Segmentation & Geometry Evaluation Metrics

In [5]:
from preprocessing_pipeline.evaluate_metrics import (
    compute_detection_precision_recall,
    compute_confusion_miou,
    compute_geometry_metrics,
)

det = compute_detection_precision_recall([[10, 10, 50, 30], [100, 120, 140, 160]], [[12, 10, 48, 29], [102, 118, 139, 159]])
seg = compute_confusion_miou({'wall': (880, 950, 940), 'bedroom': (1900, 2000, 2050), 'bathroom': (420, 460, 450)})
geo = compute_geometry_metrics([21.8, 17.6, 10.4], [22.0, 17.3, 10.6])

print('Detection Metrics (IoU>=0.5):', det)
print('Segmentation mIoU:', seg['mIoU'])
print('Geometry Room Area Error:', geo)

Detection Metrics (IoU>=0.5): {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'tp': 2, 'fp': 0, 'fn': 0}
Segmentation mIoU: 0.8712
Geometry Room Area Error: {'evaluated_rooms': 3, 'mae_sqm': 0.233, 'rel_error_pct': 1.48}
